In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Transpiler 최적화 레벨 설정

<details>
<summary><b>패키지 버전</b></summary>

이 페이지의 코드는 다음 요구 사항을 사용하여 개발되었습니다.
이 버전 이상을 사용하시기를 권장합니다.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
실제 양자 장치는 노이즈와 Gate 오류의 영향을 받으므로, Circuit의 깊이와 Gate 수를 줄이도록 최적화하면 해당 Circuit 실행 결과를 크게 향상시킬 수 있습니다.
[`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) 함수에는 필수 위치 인수인 `optimization_level`이 있으며, 이 인수는 Transpiler가 Circuit 최적화에 얼마나 많은 노력을 기울이는지 제어합니다. 이 인수는 0, 1, 2, 3 중 하나의 값을 가지는 정수입니다.
최적화 레벨이 높을수록 더 최적화된 Circuit이 생성되지만, 컴파일 시간이 더 오래 걸립니다.
다음 표는 각 설정에서 수행되는 최적화를 설명합니다.

<table>
  <thead>
    <tr>
      <th>최적화 레벨</th>
      <th>설명</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>0</td>
      <td>
        최적화 없음: 주로 하드웨어 특성 분석에 사용됩니다.
        - 기본 변환
        - 레이아웃/라우팅: `TrivialLayout` 사용. 가상 Qubit과 동일한 물리적 Qubit 번호를 선택하고 SWAP을 삽입하여 작동시킵니다 (`SabreSwap` 사용).
      </td>
    </tr>
    <tr>
      <td>1</td>
      <td>
        경량 최적화:
        -   레이아웃/라우팅: 먼저 `TrivialLayout`으로 레이아웃을 시도합니다. 추가 SWAP이 필요한 경우 `SabreSwap`을 사용하여 최소 SWAP 수의 레이아웃을 찾고, 이후 `VF2LayoutPostLayout`을 사용하여 그래프에서 최적의 Qubit을 선택하려고 시도합니다.
        -   `InverseCancellation`
        -   1Q Gate 최적화
      </td>
    </tr>
    <tr>
      <td>2</td>
      <td>
        중간 최적화:
          - 레이아웃/라우팅: 최적화 레벨 1 (trivial 제외) + 더 큰 탐색 깊이와 최적화 함수 시도 횟수로 경험적 최적화 수행. `TrivialLayout`을 사용하지 않으므로 물리적 Qubit 번호와 가상 Qubit 번호를 동일하게 사용하려는 시도가 없습니다.
        -   `CommutativeCancellation`
      </td>
    </tr>
    <tr>
      <td>3</td>
      <td>
        고수준 최적화:
        - 최적화 레벨 2 + 더 많은 노력/시도로 레이아웃/라우팅에 대한 경험적 최적화 추가 수행
        - [Cartan의 KAK 분해](https://arxiv.org/abs/quant-ph/0507171)를 사용하여 2-Qubit 블록 재합성
        - 유니타리성을 깨는 패스:
          * `OptimizeSwapBeforeMeasure`: SWAP을 피하기 위해 측정 위치를 이동합니다.
          * `RemoveDiagonalGatesBeforeMeasure`: 측정에 영향을 미치지 않는 측정 전 Gate를 제거합니다.
      </td>
    </tr>
  </tbody>
</table>

## 실제 최적화 레벨 동작
2-Qubit Gate는 일반적으로 오류의 가장 중요한 원인이므로, 결과 Circuit에서 2-Qubit Gate의 수를 세어 트랜스파일의 "하드웨어 효율성"을 대략적으로 정량화할 수 있습니다.
여기서는 무작위 유니타리에 SWAP Gate가 이어지는 입력 Circuit에 대해 다양한 최적화 레벨을 시도해 보겠습니다.

In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import Operator, random_unitary

UU = random_unitary(4, seed=12345)
rand_U = UnitaryGate(UU)

qc = QuantumCircuit(2)
qc.append(rand_U, range(2))
qc.swap(0, 1)
qc.draw("mpl", style="iqp")

<Image src="../docs/images/guides/set-optimization/extracted-outputs/81173ebc-8359-48a6-b585-0477907b3b93-0.svg" alt="Output of the previous code cell" />

![이전 코드 셀의 출력](../docs/images/guides/set-optimization/extracted-outputs/81173ebc-8359-48a6-b585-0477907b3b93-0.svg)

예제에서는 `FakeSherbrooke` 모의 Backend를 사용하겠습니다. 먼저 최적화 레벨 0으로 트랜스파일해 보겠습니다.

In [2]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()

pass_manager = generate_preset_pass_manager(
    optimization_level=0, backend=backend, seed_transpiler=12345
)
qc_t1_exact = pass_manager.run(qc)
qc_t1_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/40cdd173-b437-48b1-8928-741e8411342e-0.svg" alt="Output of the previous code cell" />

![이전 코드 셀의 출력](../docs/images/guides/set-optimization/extracted-outputs/40cdd173-b437-48b1-8928-741e8411342e-0.svg)

트랜스파일된 Circuit에는 6개의 2-Qubit ECR Gate가 있습니다.

최적화 레벨 1로 반복합니다:

In [3]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()

pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend, seed_transpiler=12345
)
qc_t1_exact = pass_manager.run(qc)
qc_t1_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/2dab5def-a017-42e9-92d6-e043ac4065b2-0.svg" alt="Output of the previous code cell" />

![이전 코드 셀의 출력](../docs/images/guides/set-optimization/extracted-outputs/2dab5def-a017-42e9-92d6-e043ac4065b2-0.svg)

트랜스파일된 Circuit에는 여전히 6개의 ECR Gate가 있지만, 단일 Qubit Gate의 수는 감소했습니다.

최적화 레벨 2로 반복합니다:

In [4]:
pass_manager = generate_preset_pass_manager(
    optimization_level=2, backend=backend, seed_transpiler=12345
)
qc_t2_exact = pass_manager.run(qc)
qc_t2_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/77d76048-b1e8-4225-b35f-80dc9d458e8d-0.svg" alt="Output of the previous code cell" />

![이전 코드 셀의 출력](../docs/images/guides/set-optimization/extracted-outputs/77d76048-b1e8-4225-b35f-80dc9d458e8d-0.svg)

이는 최적화 레벨 1과 동일한 결과를 제공합니다. 최적화 레벨을 높인다고 해서 항상 차이가 생기는 것은 아님을 주의하세요.

최적화 레벨 3으로 다시 반복합니다:

In [5]:
pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend, seed_transpiler=12345
)
qc_t3_exact = pass_manager.run(qc)
qc_t3_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/4109d0e2-df37-4850-8409-6b860c48595c-0.svg" alt="Output of the previous code cell" />

![이전 코드 셀의 출력](../docs/images/guides/set-optimization/extracted-outputs/4109d0e2-df37-4850-8409-6b860c48595c-0.svg)

이제 ECR Gate가 3개밖에 없습니다. 최적화 레벨 3에서 Qiskit은 2-Qubit Gate 블록을 재합성하려고 시도하며, 임의의 2-Qubit Gate는 최대 3개의 ECR Gate로 구현할 수 있기 때문에 이 결과를 얻게 됩니다. `approximation_degree`를 1보다 작은 값으로 설정하면 ECR Gate를 더욱 줄일 수 있으며, 이를 통해 Transpiler가 Gate 분해에 약간의 오류를 도입할 수 있는 근사를 허용합니다 ([트랜스파일에 자주 사용되는 매개변수](common-parameters#approximation-degree) 참조):

In [6]:
pass_manager = generate_preset_pass_manager(
    optimization_level=3,
    approximation_degree=0.99,
    backend=backend,
    seed_transpiler=12345,
)
qc_t3_approx = pass_manager.run(qc)
qc_t3_approx.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/bf239116-b8bb-42aa-a27a-89206d9e108a-0.svg" alt="Output of the previous code cell" />

![이전 코드 셀의 출력](../docs/images/guides/set-optimization/extracted-outputs/bf239116-b8bb-42aa-a27a-89206d9e108a-0.svg)

이 Circuit에는 ECR Gate가 2개만 있지만 근사 Circuit입니다. 이 Circuit의 효과가 정확한 Circuit과 어떻게 다른지 이해하기 위해, 이 Circuit이 구현하는 유니타리 연산자와 정확한 유니타리 사이의 충실도(fidelity)를 계산할 수 있습니다. 계산을 수행하기 전에, 먼저 127개의 Qubit을 포함하는 트랜스파일된 Circuit을 활성 Qubit만 포함하는 Circuit(총 2개)으로 축소합니다.

In [7]:
import numpy as np


def trace_to_fidelity_2q(trace: float) -> float:
    return (4.0 + trace * trace.conjugate()) / 20.0


# Reduce circuits down to 2 qubits so they are easy to simulate
qc_t3_exact_small = QuantumCircuit.from_instructions(qc_t3_exact)
qc_t3_approx_small = QuantumCircuit.from_instructions(qc_t3_approx)

# Compute the fidelity
exact_fid = trace_to_fidelity_2q(
    np.trace(np.dot(Operator(qc_t3_exact_small).adjoint().data, UU))
)
approx_fid = trace_to_fidelity_2q(
    np.trace(np.dot(Operator(qc_t3_approx_small).adjoint().data, UU))
)
print(
    f"Synthesis fidelity\nExact: {exact_fid:.3f}\nApproximate: {approx_fid:.3f}"
)

Synthesis fidelity
Exact: 1.000+0.000j
Approximate: 0.992+0.000j


Adjusting the optimization level can change other aspects of the circuit too, not just the number of ECR gates. For examples of how setting optimization level changes the layout, see [Representing quantum computers](./represent-quantum-computers).

## Next steps

<Admonition type="tip" title="Recommendations">
    - To learn more about the `generate_preset_passmanager` function, start with the [Transpilation default settings and configuration options](defaults-and-configuration-options) topic.
    - Continue learning about transpilation with the [Transpiler stages](transpiler-stages) topic.
    - Try the [Compare transpiler settings](/docs/guides/circuit-transpilation-settings) guide.
    - Try the [Build repetition codes](/docs/tutorials/repetition-codes) tutorial.
    - See the [Transpile API documentation.](/docs/api/qiskit/transpiler)
</Admonition>